# 08 · Label checks: contradictions and identical-name negatives

> **Status (2026-11-25):** the conclusions and findings in this notebook were written on the **uncleaned** pairs (`pairs_scoped.parquet`). The code now runs on the cleaned pairs (`config.PAIRS_INPUT` = `pairs_clean.parquet`); after the re-run, the numbers in the conclusions must be updated.

Two checks of the labels, both input for the cleaning (Vova) and the error taxonomy:

1. **Contradictions:** IDs linked by positive pairs form a person cluster (A = B, B = C). A **negative** pair whose two IDs lie in the same cluster contradicts that chain (A ≠ C). Same for a pair labelled both positive and negative.
2. **Identical-name negatives:** 3.7% of the negative pairs have the same full name on both sides; they make up about 80% of the false alarms at 99% recall (notebook 07). Are they really different people, or label errors? First the record fields (birth date, country, ID number) for all of them, then examples of the most suspicious group (same name and same birth date).

Input: `config.PAIRS_INPUT` (uncleaned) and `features_name.parquet`. The list of contradictions (with names) goes to `data/interim/` (gitignored), never into git.

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from screening import config as C
from screening.lists import values
from screening.name_features import normalize

pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)
props = ["name", "alias", "birthDate", "country", "nationality", "idNumber"]
cols = ["pair_id", "left_id", "right_id", "judgement", "left_schema"] + [f"{s}_{p}" for s in ("left", "right") for p in props]
pairs = pd.read_parquet(C.PAIRS_INPUT, columns=cols)
feats = pd.read_parquet(C.FEATURES_NAME, columns=["pair_id", "n_exact_name"])
df = pairs.merge(feats, on="pair_id")
df = df[df["judgement"].isin(["positive", "negative"])].reset_index(drop=True)
df["y"] = (df["judgement"] == "positive").astype(int)
df["person"] = df["left_schema"] == "Person"
print(f"{len(df):,} labelled pairs, {(df['y'] == 0).sum():,} negative")

## 1 · Contradictions

Union-find over the positive pairs gives the person clusters (as in notebook 07, section 10). Three kinds of contradiction:

- **negative self-pair:** an ID compared with itself and labelled negative
- **same pair, both labels:** the same two IDs (in either order) appear with a positive and a negative label
- **negative inside a cluster:** a negative pair whose IDs are linked by a chain of positive pairs

In [ ]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in zip(df.loc[df["y"] == 1, "left_id"], df.loc[df["y"] == 1, "right_id"]):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
df["cl_l"], df["cl_r"] = df["left_id"].map(find), df["right_id"].map(find)
csize = pd.concat([df["left_id"], df["right_id"]]).drop_duplicates().map(find).value_counts()

neg = df[df["y"] == 0]
self_neg = neg["left_id"] == neg["right_id"]
key = pd.Series([tuple(sorted(t)) for t in zip(df["left_id"], df["right_id"])], index=df.index)
both = df.assign(key=key).groupby("key")["y"].agg(["min", "max"])
both_keys = set(both[(both["min"] == 0) & (both["max"] == 1)].index)
same_pair_both = neg.index.isin(key[key.isin(both_keys)].index)
inside = (neg["cl_l"] == neg["cl_r"]) & ~self_neg

contra = pd.DataFrame({
    "negative pairs": [len(neg)] * 3,
    "contradictions": [int(self_neg.sum()), int(same_pair_both.sum()), int(inside.sum())],
}, index=["negative self-pair", "same pair, both labels", "negative inside a cluster (all)"])
contra["share of negatives"] = contra["contradictions"] / contra["negative pairs"]
contra

**Conclusion.**
- **Contradictions are negligible:** 9 negative self-pairs, 10 pairs with both labels and 12 negatives inside a person cluster, out of 164,156 negatives (each below 0.01%).
- **The labels are consistent** enough for person clusters to be used as the unit of the split.

**Next step:** remove these pairs in the cleaning (Vova).

### Table: contradictions

**What it shows:** for each kind, the number of negative pairs and their share of all negative pairs. "Negative inside a cluster" includes the "same pair, both labels" cases that are not self-pairs.

**How to read it:** these pairs cannot all be right. A few hundred would be noise; thousands would mean that either some positive labels are wrong (two people merged by mistake) or some negative labels are (one person split). Either way they need a rule in the cleaning.

In [ ]:
ins = neg[inside].copy()
ins["cluster_size"] = ins["cl_l"].map(csize)
ins["same pair, both labels"] = same_pair_both[inside.to_numpy()]
t = pd.DataFrame({
    "pairs": [len(ins)],
    "persons": [ins["person"].mean()],
    "identical name": [(ins["n_exact_name"] == 1).mean()],
    "also labelled positive (same pair)": [ins["same pair, both labels"].mean()],
    "clusters affected": [ins["cl_l"].nunique()],
    "median cluster size (IDs)": [ins["cluster_size"].median()],
}, index=["negative inside a cluster"]).T
t

**Conclusion.**
- **The 12 negatives inside a cluster** lie in larger clusters (median 15.5 IDs), are mostly organisations (only a third persons) and two thirds have identical names.
- **Likely cause:** one long chain of positive labels merged two different organisations, or the negative is wrong. A quarter are the same pair also labelled positive.

**Next step:** list them for Vova (`data/interim/label_contradictions.csv`); dropping them is enough, too few to matter for the results.

### Table: what the contradicting negatives look like

**What it shows:** for the negatives inside a person cluster: share of persons, share with identical names, share that are the same pair also labelled positive, number of clusters affected and their median size.

**How to read it:** if most have identical names and lie in larger clusters, a chain of positive labels probably merged two people (one wrong positive in the chain), or the negative is wrong; the list below lets Vova look at them.

In [ ]:
first = lambda v: list(v)[0] if v is not None and len(v) else ""
out = ins.assign(left_name_1=ins["left_name"].map(first), right_name_1=ins["right_name"].map(first))
out = out[["pair_id", "left_id", "right_id", "left_schema", "left_name_1", "right_name_1", "n_exact_name",
           "cluster_size", "same pair, both labels"]]
path = C.DATA_INTERIM / "label_contradictions.csv"
out.to_csv(path, index=False, encoding="utf-8-sig")
print(f"wrote {path.name}: {len(out):,} rows")
out.sort_values("cluster_size").head(10)

## 2 · Identical-name negatives: what the record fields say

For every negative pair with the same full name (n_exact_name = 1, word order and transliteration ignored), compare the record fields of both sides:

- **same:** a value occurs on both sides (birth dates also count as same if one is a prefix of the other, e.g. 1970 vs 1970-05-01)
- **different:** both sides have values, none match
- **missing:** at least one side has no value

Country and nationality are taken together. A pair with identical name **and** same birth date labelled negative is a strong candidate for a label error.

In [ ]:
def status(r, props, prefix_ok=False):
    L = {normalize(v) for v in values(r, "left", props)} - {""}
    R = {normalize(v) for v in values(r, "right", props)} - {""}
    if not L or not R:
        return "missing"
    if L & R:
        return "same"
    if prefix_ok and any(a.startswith(b) or b.startswith(a) for a in L for b in R):
        return "same"
    return "different"

idn = neg[neg["n_exact_name"] == 1].copy()
idn["birth date"] = [status(r, ["birthDate"], prefix_ok=True) for r in idn.itertuples(index=False)]
idn["country"] = [status(r, ["country", "nationality"]) for r in idn.itertuples(index=False)]
idn["ID number"] = [status(r, ["idNumber"]) for r in idn.itertuples(index=False)]
print(f"{len(idn):,} identical-name negatives ({idn['person'].mean():.0%} persons)")
summary = pd.concat({f: idn.groupby("person")[f].value_counts().unstack(fill_value=0)
                     for f in ["birth date", "country", "ID number"]}, axis=1)
summary.index = summary.index.map({True: "persons", False: "companies"})
summary

**Conclusion.**
- **Persons (5,220 pairs):** for 86% (4,510) the birth date is missing on at least one side. Only 561 have different birth dates, which a birth-date feature can use, and 149 have the same birth date.
- **ID numbers:** missing in 69% of the person pairs and 98% of the company pairs. Where both sides have one, they mostly differ (1,623), which supports the negative label.
- **So record features can separate only a minority of these pairs.**

**Next step:** cross birth date with country (next table).

### Table: record fields of the identical-name negatives

**What it shows:** for persons and companies separately, how many identical-name negatives have the same / different / missing birth date, country (incl. nationality) and ID number.

**How to read it:** "different" supports the label (same name, different people: the floor for name features, but record features can separate them). "same" on the birth date or ID number makes the negative label doubtful. "missing" means nothing can separate the pair, neither for us nor for any model.

In [ ]:
p = idn[idn["person"]]
combo = p.groupby(["birth date", "country"]).size().unstack(fill_value=0)
combo["total"] = combo.sum(axis=1)
combo.loc["total"] = combo.sum()
combo

**Conclusion.**
- **2,061 person pairs (39%) have neither a birth date nor a country on both sides:** nothing in the data separates them, for any model.
- **2,327 pairs have no birth date but the same country,** and they are just as hard.
- **The 149 with the same birth date are the label-error candidates;** 53 of them also share the country.

**Next step:** read the sample below; for the report, state the floor (pairs no feature can separate).

### Table: persons, birth date × country

**What it shows:** the identical-name negatives between persons, split by birth date status (rows) and country status (columns).

**How to read it:** the top-left corner (same birth date, same country) is the most suspicious group. "different birth date" is the group where Vova's record features will remove most false alarms.

## 3 · Examples: same name and same birth date

15 of the 149 identical-name negatives between persons that also share the **birth date**: the pairs whose negative label is most doubtful. Shown with the names and birth dates of both sides; nothing is written to disk.

**How to read it:** pairs that are the same person in another spelling or script point to label errors; pairs with different main names that share only one alias are probably different people.

In [ ]:
smp = idn[idn["person"] & (idn["birth date"] == "same")].sample(15, random_state=7)
join = lambda v, n=8: " | ".join(list(v)[:n]) if v is not None and len(v) else ""
ex = pd.DataFrame({"schema": smp["left_schema"], "status birth date": smp["birth date"]})
for s in ("left", "right"):
    ex[f"{s} name"] = smp[f"{s}_name"].map(join)
    ex[f"{s} birthDate"] = smp[f"{s}_birthDate"].map(join)
ex[["schema", "status birth date", "left name", "right name", "left birthDate", "right birthDate"]]

**Conclusion.**
- **Several same-birth-date pairs look like the same person** in another spelling or script (Tereshkova, Pogorelov, Potapov, Aung Moe Myint, Torres-Lopez), so they are probably label errors.
- **Others share a birth date but have completely different main names** (several Iraqi entries); they match only through a shared alias and are probably different people.

**Next step:** none. The 149 pairs are reported in the error taxonomy as label-error candidates; the data cannot settle them.

## Findings (uncleaned data, 2026-11-18)

**Contradictions: negligible.**
- 9 negative self-pairs, 10 pairs labelled both positive and negative, 12 negatives inside a person cluster (in 10 clusters), out of 164,156 negatives.
- The positive chains are therefore clean: a split by person cluster rests on consistent labels.
- For the cleaning: drop these 31 pairs or treat them by one rule. The list is in `data/interim/label_contradictions.csv`.

**Identical-name negatives: 6,034, 87% persons.**
- **Most cannot be separated by record fields either.** For 4,510 of the 5,220 person pairs (86%) the birth date is missing on at least one side; 2,061 have neither birth date nor country on both sides. For companies the ID number is missing in 798 of 814 pairs.
  - These pairs stay false alarms even with Vova's features: a floor for any model on this data.
- **561 person pairs have different birth dates.** That is where a birth-date feature can remove false alarms (about 11% of the person pairs).
- **149 person pairs have the same birth date: label-error candidates.**
  - In the sample several look like the same person under another spelling or script (e.g. Tereshkova, Aung Moe Myint, Potapov, Pogorelov, Torres-Lopez).
- **"Identical name" also catches pairs that share only one alias.** Some sample pairs have completely different main names (e.g. several Iraqi entries listed together) but one identical alias. These are probably different people with a shared or generic alias.
  - n_exact_name = 1 therefore means "at least one identical name or alias", not "the same main name".
